In [4]:
!pip install langchain
!pip install langchain_groq
!pip install langchain-community
!pip install chromadb

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.8/108.8 kB 2.7 MB/s eta 0:00:00


In [5]:
from langchain_groq import ChatGroq

In [6]:
llm = ChatGroq(
    temperature=0,
    groq_api_key='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',  # It a Demo API Key (You need to replace with your own API Key)
    model_name="llama-3.1-70b-versatile"
)
response = llm.invoke("The first person to land on moon was ...")
print(response.content)

The first person to land on the moon was Neil Armstrong. He stepped onto the lunar surface on July 20, 1969, as part of the Apollo 11 mission.


In [7]:
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://jobs.nike.com/job/R-45775") # Lets consider we are searching for data engineer job role
page_data = loader.load().pop().page_content
print(page_data)

Apply for Data Engineer

Search JobsSkip navigationSearch JobsNIKE, INC. JOBSContract JobsJoin The Talent CommunityLife @ NikeOverviewBenefitsBrandsOverviewJordanConverseTeamsOverviewAdministrative SupportAdvanced InnovationAir Manufacturing InnovationAviationCommunicationsCustomer ServiceDesignDigitalFacilitiesFinance & AccountingGovernment & Public AffairsHuman ResourcesInsights & AnalyticsLegalManufacturing & EngineeringMarketingMerchandisingPlanningPrivacyProcurementProduct Creation, Development & ManagementRetail CorporateRetail StoresSalesSocial & Community ImpactSports MarketingStrategic PlanningSupply Chain, Distribution & LogisticsSustainabilityTechnologyLocationsOverviewNike WHQNike New York HQEHQ: Hilversum, The NetherlandsELC: Laakdal, BelgiumGreater China HQDiversity, Equity & InclusionOverviewMilitary InclusionDisability InclusionIndigenous InclusionInternshipsData & AnalyticsData EngineerBeaverton, OregonBecome a Part of the NIKE, Inc. Team
NIKE, Inc. does more than outf

In [8]:
from langchain_core.prompts import PromptTemplate

prompt_extract = PromptTemplate.from_template(
        """
        ### SCRAPED TEXT FROM WEBSITE:
        {page_data}
        ### INSTRUCTION:
        The scraped text is from the career's page of a website.
        Your job is to extract the job postings and return them in JSON format containing the
        following keys: `role`, `experience`, `skills` and `description`.
        Only return the valid JSON.
        ### VALID JSON (NO PREAMBLE):
        """
)

chain_extract = prompt_extract | llm
res = chain_extract.invoke(input={'page_data':page_data})
print(res.content)

```json
{
  "role": "Data Engineer",
  "experience": "2 years",
  "skills": [
    "Python",
    "SQL",
    "Apache Hadoop",
    "Apache Hive",
    "AWS",
    "Airflow",
    "Databricks",
    "Apache Spark",
    "Teradata",
    "Snowflake",
    "Kinesis",
    "Lambda",
    "EMR"
  ],
  "description": "Design and implement features in collaboration with product owners, data analysts, and business partners using Agile / Scrum methodology; contribute to overall architecture, frameworks and patterns for processing and storing large data volumes; design and implement distributed data processing pipelines using tools and languages prevalent in the Hadoop or Cloud ecosystems; build utilities, user defined functions, and frameworks to better enable data flow patterns; build and develop job orchestration and scheduling using Airflow; research, evaluate and utilize new technologies/tools/frameworks centered around high-volume data processing; define and apply appropriate data acquisition and cons

In [9]:
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()
json_res = json_parser.parse(res.content)
json_res

{'role': 'Data Engineer',
 'experience': '2 years',
 'skills': ['Python',
  'SQL',
  'Apache Hadoop',
  'Apache Hive',
  'AWS',
  'Airflow',
  'Databricks',
  'Apache Spark',
  'Teradata',
  'Snowflake',
  'Kinesis',
  'Lambda',
  'EMR'],
 'description': 'Design and implement features in collaboration with product owners, data analysts, and business partners using Agile / Scrum methodology; contribute to overall architecture, frameworks and patterns for processing and storing large data volumes; design and implement distributed data processing pipelines using tools and languages prevalent in the Hadoop or Cloud ecosystems; build utilities, user defined functions, and frameworks to better enable data flow patterns; build and develop job orchestration and scheduling using Airflow; research, evaluate and utilize new technologies/tools/frameworks centered around high-volume data processing; define and apply appropriate data acquisition and consumption strategies for given technical scenari

In [10]:
type(json_res)

dict

In [11]:
# For Mounting your google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [12]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/Colab_Notebooks/Gen_AI_Latest_Projects/project-genai-cold-email-generator-main/my_portfolio.csv")
df

,Techstack,Links
0,"React, Node.js, MongoDB",https://example.com/react-portfolio
1,"Angular,.NET, SQL Server",https://example.com/angular-portfolio
2,"Vue.js, Ruby on Rails, PostgreSQL",https://example.com/vue-portfolio
3,"Python, Django, MySQL",https://example.com/python-portfolio
4,"Java, Spring Boot, Oracle",https://example.com/java-portfolio
5,"Flutter, Firebase, GraphQL",https://example.com/flutter-portfolio
6,"WordPress, PHP, MySQL",https://example.com/wordpress-portfolio
7,"Magento, PHP, MySQL",https://example.com/magento-portfolio
8,"React Native, Node.js, MongoDB",https://example.com/react-native-portfolio
9,"iOS, Swift, Core Data",https://example.com/ios-portfolio


No charts were generated by quickchart


In [13]:
import uuid
import chromadb

client = chromadb.PersistentClient('vectorstore')
collection = client.get_or_create_collection(name="portfolio")

if not collection.count():
    for _, row in df.iterrows():
        collection.add(documents=row["Techstack"],
                       metadatas={"links": row["Links"]},
                       ids=[str(uuid.uuid4())])

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 77.9MiB/s]


In [14]:
job = json_res
job['skills']

['Python',
 'SQL',
 'Apache Hadoop',
 'Apache Hive',
 'AWS',
 'Airflow',
 'Databricks',
 'Apache Spark',
 'Teradata',
 'Snowflake',
 'Kinesis',
 'Lambda',
 'EMR']

In [15]:
links = collection.query(query_texts=job['skills'], n_results=2).get('metadatas', [])
links

[[{'links': 'https://example.com/ml-python-portfolio'},
  {'links': 'https://example.com/python-portfolio'}],
 [{'links': 'https://example.com/magento-portfolio'},
  {'links': 'https://example.com/wordpress-portfolio'}],
 [{'links': 'https://example.com/magento-portfolio'},
  {'links': 'https://example.com/wordpress-portfolio'}],
 [{'links': 'https://example.com/magento-portfolio'},
  {'links': 'https://example.com/java-portfolio'}],
 [{'links': 'https://example.com/ios-ar-portfolio'},
  {'links': 'https://example.com/wordpress-portfolio'}],
 [{'links': 'https://example.com/ml-python-portfolio'},
  {'links': 'https://example.com/ios-ar-portfolio'}],
 [{'links': 'https://example.com/flutter-portfolio'},
  {'links': 'https://example.com/kotlin-android-portfolio'}],
 [{'links': 'https://example.com/vue-portfolio'},
  {'links': 'https://example.com/kotlin-android-portfolio'}],
 [{'links': 'https://example.com/android-tv-portfolio'},
  {'links': 'https://example.com/ml-python-portfolio'}],


In [16]:
job

{'role': 'Data Engineer',
 'experience': '2 years',
 'skills': ['Python',
  'SQL',
  'Apache Hadoop',
  'Apache Hive',
  'AWS',
  'Airflow',
  'Databricks',
  'Apache Spark',
  'Teradata',
  'Snowflake',
  'Kinesis',
  'Lambda',
  'EMR'],
 'description': 'Design and implement features in collaboration with product owners, data analysts, and business partners using Agile / Scrum methodology; contribute to overall architecture, frameworks and patterns for processing and storing large data volumes; design and implement distributed data processing pipelines using tools and languages prevalent in the Hadoop or Cloud ecosystems; build utilities, user defined functions, and frameworks to better enable data flow patterns; build and develop job orchestration and scheduling using Airflow; research, evaluate and utilize new technologies/tools/frameworks centered around high-volume data processing; define and apply appropriate data acquisition and consumption strategies for given technical scenari

In [17]:
prompt_email = PromptTemplate.from_template(
        """
        ### JOB DESCRIPTION:
        {job_description}

        ### INSTRUCTION:
        You are Mohan, a business development executive at AtliQ. AtliQ is an AI & Software Consulting company dedicated to facilitating
        the seamless integration of business processes through automated tools.
        Over our experience, we have empowered numerous enterprises with tailored solutions, fostering scalability,
        process optimization, cost reduction, and heightened overall efficiency.
        Your job is to write a cold email to the client regarding the job mentioned above describing the capability of AtliQ
        in fulfilling their needs.
        Also add the most relevant ones from the following links to showcase Atliq's portfolio: {link_list}
        Remember you are Mohan, BDE at AtliQ.
        Do not provide a preamble.
        ### EMAIL (NO PREAMBLE):

        """
        )

chain_email = prompt_email | llm
res = chain_email.invoke({"job_description": str(job), "link_list": links})
print(res.content)

Subject: Expert Data Engineering Solutions for Your Business Needs

Dear [Client's Name],

I came across your job description for a Data Engineer and was impressed by the scope of the role. As a Business Development Executive at AtliQ, I'd like to introduce you to our team of experts who can help you design and implement features that meet your business needs.

At AtliQ, we have extensive experience in facilitating seamless integration of business processes through automated tools. Our team of data engineers has expertise in designing and implementing distributed data processing pipelines using tools and languages prevalent in the Hadoop or Cloud ecosystems. We're well-versed in Apache Hadoop, Apache Hive, AWS, Airflow, Databricks, Apache Spark, Teradata, Snowflake, Kinesis, Lambda, and EMR.

Our portfolio showcases our capabilities in delivering high-quality solutions. Some of our notable projects include:

* Machine Learning and Python-based solutions: https://example.com/ml-python-p